# Lesson 1 — The PTC Schema: Typed Causal Claims

**Learning objective:** Understand what a Point To Correlate (PTC) is, why every field exists, and the hard-won distinctions (`source_date` vs `event_date`, direction anchoring, atomicity) that make downstream analysis valid.

---

## What is a PTC?

A **PTC (Point To Correlate)** is the atomic unit of this pipeline. It captures one directional, signed causal claim extracted from a corporate document:

> *Red Sea disruptions added €45M in logistics costs over the first nine months and management does not expect resolution before mid-2025.*

From this one sentence, a careful analyst would note:
- **Mechanism**: elevated logistics costs due to Red Sea disruption
- **Direction**: precursor — the disruption is *ongoing* with forward signal
- **Polarity**: −1 — it pushes financial performance *down*
- **source_date**: 2024-11-14 (when this document was published)
- **event_date**: 2024-01-01 (disruptions began in early 2024)

This precision is what separates PTCs from sentiment scores.

---

## The schema

In [ ]:
import sys
sys.path.insert(0, '../../src')

from ptc_extraction.schema import PTC, Direction, Polarity, SourceType

print("Fields on PTC:")
for name, field in PTC.model_fields.items():
    print(f"  {name:<25} {field.annotation}")

## Exercise 1 — Construct a valid PTC

Let's create the Red Sea PTC from the sentence above.

In [ ]:
from datetime import date

ptc = PTC(
    mechanism=(
        "Red Sea disruptions elevated logistics costs by approximately €45M over nine months "
        "and are expected to persist into mid-2025, continuing to pressure operating margins."
    ),
    raw_text=(
        "Red Sea disruptions added €45M in logistics costs over the first nine months "
        "and management does not expect resolution before mid-2025."
    ),
    direction=Direction.PRECURSOR,    # ongoing → still carries forward signal
    polarity=Polarity.NEGATIVE,        # pushes margins DOWN
    source_date=date(2024, 11, 14),    # document publication date
    event_date=date(2024, 1, 1),       # disruptions began early 2024 → "early YYYY" → YYYY-02-01
    source_type=SourceType.MANAGEMENT,
    source_id="ML.PA-management-2024-11-14-q3-results",
    extracted_by="eu.anthropic.claude-sonnet-4-6",
    extraction_version="v1.1",
    confidence=0.92,
)

print(ptc.model_dump_json(indent=2))

## The `source_date` vs `event_date` distinction

This is the most critical design decision in the schema.

| Field | Meaning | Used for |
|-------|---------|----------|
| `source_date` | When the **document was published** — always known, always tradable | Predictive analysis (Granger, event study) |
| `event_date` | When the **underlying event occurred** — may be in the past | Causal attribution analysis |

**Why does this matter?** A 2024 annual report can mention a 2022 supply shock. If you anchor your Granger analysis on `event_date=2022`, you introduce forward-looking bias — you'd be "predicting" something the market already knew two years ago. Always anchor on `source_date`.

In [ ]:
# event_date must be <= source_date — the validator enforces this
from pydantic import ValidationError

try:
    bad_ptc = PTC(
        mechanism="A future event mechanism that is long enough to pass validation",
        raw_text="some raw text",
        direction=Direction.PRECURSOR,
        polarity=Polarity.POSITIVE,
        source_date=date(2024, 1, 1),
        event_date=date(2025, 6, 1),   # AFTER source_date — invalid!
        source_type=SourceType.MANAGEMENT,
        source_id="test-id",
        extracted_by="test",
        extraction_version="v1.1",
    )
except ValidationError as e:
    print("Validation caught it:")
    print(e)

## The direction rule — anchored on `source_date`

Direction is NOT about whether an event is in the past. It is about whether the **financial impact** still carries forward signal as of the document's publication date.

| Classification | Meaning | Example |
|---------------|---------|--------|
| `precursor` | Impact expected to manifest at or after `source_date` | New tariff effective next month; ongoing disruption |
| `consequence` | Impact fully realised before `source_date`, no residual signal | One-off Q2 restructuring charge now closed |

**The trap**: an ongoing mechanism that *started* in the past but *continues* into the future is a **precursor**, not a consequence. The Red Sea example above is the canonical case.

In [ ]:
# Exercise: classify these three mechanisms
scenarios = [
    {
        "text": "The Q2 restructuring charge of €30M is now fully reflected in our accounts.",
        "correct_direction": Direction.CONSEQUENCE,
        "rationale": "Fully realised, closed. No forward signal.",
    },
    {
        "text": "EU tariffs effective October 30 are expected to improve our competitive positioning.",
        "correct_direction": Direction.PRECURSOR,
        "rationale": "Future impact — not yet in P&L.",
    },
    {
        "text": "Rubber price surge is now visible in Q3 and is expected to persist through year-end.",
        "correct_direction": Direction.PRECURSOR,
        "rationale": "Ongoing: partially realised AND continuing. Forward signal remains.",
    },
]

for s in scenarios:
    print(f"Text : {s['text'][:70]}...")
    print(f"  → {s['correct_direction'].value:12s} | {s['rationale']}")
    print()

## Atomicity — one mechanism per PTC

A passage that mentions both a **realised** impact AND an **expected future** impact from the same cause must produce **two** PTCs. Fusing them would contaminate the temporal analysis.

In [ ]:
# Forex example — must be split into two PTCs
passage = (
    "The stronger euro compressed Q3 revenue by €120M; "
    "forex headwinds are expected to persist into Q4."
)

ptc_consequence = PTC(
    mechanism="Euro appreciation realised a €120M negative translation impact on Q3 revenue.",
    raw_text=passage,
    direction=Direction.CONSEQUENCE,
    polarity=Polarity.NEGATIVE,
    source_date=date(2024, 11, 14),
    event_date=date(2024, 7, 1),   # Q3 2024
    source_type=SourceType.MANAGEMENT,
    source_id="ML.PA-management-2024-11-14-q3-results",
    extracted_by="test",
    extraction_version="v1.1",
)

ptc_precursor = PTC(
    mechanism="Euro forex headwinds are expected to persist into Q4, continuing to compress revenue translation.",
    raw_text=passage,
    direction=Direction.PRECURSOR,
    polarity=Polarity.NEGATIVE,
    source_date=date(2024, 11, 14),
    event_date=None,   # future — no event_date
    source_type=SourceType.MANAGEMENT,
    source_id="ML.PA-management-2024-11-14-q3-results",
    extracted_by="test",
    extraction_version="v1.1",
)

print("Two PTCs from one passage:")
for p in [ptc_consequence, ptc_precursor]:
    print(f"  direction={p.direction.value:12s} polarity={p.polarity:+d}  {p.mechanism[:60]}...")

## The `content_hash` — deduplication key

In [ ]:
print("Consequence PTC hash:", ptc_consequence.content_hash())
print("Precursor  PTC hash:", ptc_precursor.content_hash())
print()
print("Different hashes → no dedup collision between the two atomised PTCs.")

---

## Key takeaways

1. A PTC = one directional, signed causal claim. Not a topic, not a sentiment score.
2. `source_date` is the tradable anchor. Never use `event_date` for Granger or event studies.
3. `direction` is not past/future — it is "does this still carry forward signal at `source_date`?" Ongoing = precursor.
4. Atomicity is not optional. Fused PTCs produce biased time series.
5. `content_hash()` enables resume-safe processing and deduplication.

**Next:** Lesson 2 — Chunking documents to feed the extractor.